# Install Libraries

In [ ]:
%%writefile requirements.txt
# Data
pandas==3.0.6
pyarrow==25.0.1
duckdb==1.5.5
# GCP
google-cloud-bigquery==3.45.2
google-cloud-storage==3.14.1
# Util
openpyxl==3.1.5
python-dotenv==1.2.3

In [ ]:
!pip install --no-cache-dir -r requirements.txt

In [ ]:
import os
os.remove('requirements.txt')

# Get ENV variable

In [ ]:
# For Local DEV
from dotenv import load_dotenv
load_dotenv()

# Get Variable
import os
GCS_PROJECT= os.getenv('GCS_PROJECT')
GCS_BUCKET= os.getenv('GCS_BUCKET')
GCS_FILE_PATH= os.getenv('GCS_FILE_PATH')
BQ_TARGET_PROJECT= os.getenv('BQ_TARGET_PROJECT')
BQ_TARGET_SCHEMA= os.getenv('BQ_TARGET_SCHEMA')
BQ_TARGET_TABLE= os.getenv('BQ_TARGET_TABLE')

# Modify ENV
LOCAL_PATH= os.path.split(GCS_FILE_PATH)[-1]

# Get Raw File

In [ ]:
from google.cloud import storage

def download( gcs_bucket, bucket_file, local_file):
    if os.getenv('LOCAL_TEST') :  os.environ["GOOGLE_APPLICATION_CREDENTIALS"]= 'key.json'
    client = storage.Client()
    bucket = client.bucket(gcs_bucket)
    blob = bucket.blob(bucket_file)
    blob.download_to_filename(local_file)
    if os.getenv('LOCAL_TEST') :  _= os.environ.pop("GOOGLE_APPLICATION_CREDENTIALS")
    
download(gcs_bucket= GCS_BUCKET, bucket_file= GCS_FILE_PATH, local_file= LOCAL_PATH)

# Data Processing

## Read File

In [ ]:
import pandas as pd

SHEET_NAME = "Raw Data"
raw = pd.read_excel(LOCAL_PATH, sheet_name=SHEET_NAME, header=None)
print(f"Raw shape: {raw.shape}")
raw.iloc[10:15, :20]

## Detect Starting Block

In [ ]:
STARTING_BLOCK= raw[raw[0].notnull()].index.min()
print(f"Starting Block at row number : {STARTING_BLOCK + 1}")

## Get Blocks and locations

In [ ]:
BLOCK_DICT= {k : v for k,v in raw.loc[STARTING_BLOCK].dropna().to_dict().items()}
BLOCK_DICT

In [ ]:
BLOCK_DICT= {k : v if k == 0 else v.replace(day= 1) for k, v in BLOCK_DICT.items()}
BLOCK_DICT

## Create data by blocks

In [ ]:
# Filter data rows
pre_prep= raw.loc[STARTING_BLOCK+1:].reset_index(drop= True)
pre_prep= pre_prep[pre_prep[0] != 'AVG']
pre_prep.head()

In [ ]:
# Prep
base_col= BLOCK_DICT[0]
col_num_list= list(BLOCK_DICT.keys())
base_date_list= list(BLOCK_DICT.values())[1:]
run_list= zip(
                col_num_list[1:], 
                col_num_list[2:] + [max(pre_prep.columns) + 1], 
                base_date_list
              )

def processing_block(base_col, n, next_n, base_date) :
    # List target Columns by number
    _target_columns= [0] + list(range(n, next_n))
    # Slide DF
    _df= pre_prep.loc[1:, _target_columns]
    # Get Header
    _header= pre_prep.loc[0, _target_columns].str.lower().to_list()[1:]
    _df.columns= [base_col.lower()] + _header
    # Drop Ref
    _df= _df.drop('reference', axis= 1)
    # Add Date
    _df['date'] = _df['date'].apply(lambda x : base_date + pd.Timedelta(days= x - 1))
    # Melt
    _df= _df.melt(id_vars= 'date', var_name= 'asset', value_name= 'nomination')
    return _df

# Run
df= []
for n, next_n, base_date in run_list :
    print(f"Block : {n}, DATE : {base_date}")
    df.append(processing_block(base_col= base_col, n= n, next_n= next_n, base_date= base_date))
    
df= pd.concat(df, axis= 0, ignore_index= True)
df= df[df['nomination'].notnull()]
df.head()

## Add Suffix

In [ ]:
# Get Local path
suffix= LOCAL_PATH.split('.')[0].split('_')[-1]
df['parameter']= pd.to_datetime(suffix, format= '%Y%m%d')

# For BQ Sandbox, has only 60 days expiration, need to ramp up
if os.getenv('LOCAL_TEST') : 
    from dateutil.relativedelta import relativedelta 
    df['parameter']= df['parameter'].apply(lambda x : x + relativedelta(years= 2))
    
df.head()

## Add Run time

In [ ]:
df["load_ts"] = pd.Timestamp.now()
df.head()

# GBQ Loading Function

## Prep Schema

In [ ]:
schema_setup= [
    {
        "name": "date",
        "type": "DATE",
        "description": "Represents the specific calendar day for which the operational data was recorded",
        "mode": "NULLABLE"
    },
    {
        "name": "parameter",
        "type": "DATE",
        "description": "This column specifically captures the date suffix extracted directly from the raw data filename",
        "mode": "NULLABLE"
    },
    {
        "name": "asset",
        "type": "STRING",
        "description": "Specifies the particular oil and gas asset or equipment from which the data originated",
        "mode": "NULLABLE"
    },
    {
        "name": "nomination",
        "type": "FLOAT",
        "description": "This refers to the nominated or planned quantity/value for the specific date of each asset",
        "mode": "NULLABLE"
    },
    {
        "name": "load_ts",
        "type": "TIMESTAMP",
        "description": "A timestamp indicating when this particular data record was loaded into the database (current timestamp)",
        "mode": "NULLABLE"
    }
]

import json
with open('schema.json', 'w') as f :
    f.write(json.dumps(schema_setup, indent= 4))

## Ensure Data Type

In [ ]:
import duckdb

# Create Query
duckdb_col= ' ,\n\t'.join([f"{x['name']}::{x['type']} AS {x['name']}" for x in schema_setup])
query= f"SELECT \n\t{duckdb_col} \nFROM df"
print(query)
    
# Ensure Data Type
duckdb.query(query).write_csv("output.csv", sep= '|', quotechar= '"')

## Load

In [ ]:
from google.cloud import bigquery

# Create Connection
client= bigquery.Client(BQ_TARGET_PROJECT)
table_full_name= f'{BQ_TARGET_PROJECT}.{BQ_TARGET_SCHEMA}.{BQ_TARGET_TABLE}'

# Create job_config
## schema
schema= client.schema_from_json('schema.json')
## partition
type_ = bigquery.TimePartitioningType.MONTH
time_partitioning= bigquery.TimePartitioning(field= 'parameter', type_= type_)
## into job_config
job_config = bigquery.LoadJobConfig(
                                    schema= schema, 
                                    write_disposition="WRITE_TRUNCATE",
                                    time_partitioning= time_partitioning,
                                    source_format = bigquery.SourceFormat.CSV,
                                    clustering_fields=["date"],
                                    skip_leading_rows=1,
                                    quote_character= '"',
                                    field_delimiter="|"
                                    )

with open('output.csv', "rb") as source_file:
    # Run a load job, with job_config
    job = client.load_table_from_file(   source_file, 
                                        f"{table_full_name}",
                                        job_config=job_config)
# Wait until the job finish
job.result()

# Clean up

In [ ]:
to_remove= ['schema.json','output.csv',LOCAL_PATH]
for i in to_remove : os.remove(i)